# BTS + SFDPS gold-flight counts for 2026-03-01

This follows `src/flights/ground_truth/bts_and_sfdps_to_gold.py`, then independently applies `get_matching_icaos_in_flights` using ADS-B data from adsb.lol, OpenSky, and ADS-B Exchange.

The network cell reads only the columns used by the matching function. It is still expensive to rerun, particularly for OpenSky.

In [1]:
from datetime import date, datetime, timedelta

import polars as pl

from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights.evaluation.american_evals import continental_usa_to_continental_usa
from flights.flights_comparison import df_flights_comparision
from flights.flights_match_adsb import ADSB_MATCHING_COLUMNS, get_matching_icaos_in_flights

target_date = date(2026, 3, 1)
start_dt = datetime.combine(target_date, datetime.min.time())
end_dt = start_dt + timedelta(days=1)

df_bts_raw = get_bts_flights_for_day(target_date)
df_sfdps_raw = get_sfdps_flights_day(target_date)

pl.DataFrame({
    "source": ["BTS", "SFDPS"],
    "flights": [df_bts_raw.height, df_sfdps_raw.height],
    "icaos": [df_bts_raw["icao"].n_unique(), df_sfdps_raw["icao"].n_unique()],
})

shape: (2, 3)
┌────────┬─────────┬───────┐
│ source ┆ flights ┆ icaos │
│ ---    ┆ ---     ┆ ---   │
│ str    ┆ i64     ┆ i64   │
╞════════╪═════════╪═══════╡
│ BTS    ┆ 17609   ┆ 4718  │
│ SFDPS  ┆ 27486   ┆ 12559 │
└────────┴─────────┴───────┘

## Apply the script's preprocessing

Keep shared ICAOs, flights landing on the target UTC date, and continental-U.S.-to-continental-U.S. flights.

In [2]:
def stage_row(stage, bts, sfdps):
    return {
        "stage": stage,
        "bts_flights": bts.height,
        "bts_icaos": bts["icao"].n_unique(),
        "sfdps_flights": sfdps.height,
        "sfdps_icaos": sfdps["icao"].n_unique(),
    }

stages = [stage_row("raw", df_bts_raw, df_sfdps_raw)]

shared_icaos = set(df_bts_raw["icao"].unique()) & set(df_sfdps_raw["icao"].unique())
df_bts_shared = df_bts_raw.filter(pl.col("icao").is_in(shared_icaos))
df_sfdps_shared = df_sfdps_raw.filter(pl.col("icao").is_in(shared_icaos))
stages.append(stage_row("shared ICAOs", df_bts_shared, df_sfdps_shared))

df_bts_day = df_bts_shared.filter(pl.col("landing_time").dt.date() == target_date)
df_sfdps_day = df_sfdps_shared.filter(pl.col("landing_time").dt.date() == target_date)
stages.append(stage_row("landing date", df_bts_day, df_sfdps_day))

df_bts_flights = continental_usa_to_continental_usa(df_bts_day)
df_sfdps_flights = continental_usa_to_continental_usa(df_sfdps_day)
stages.append(stage_row("continental U.S.", df_bts_flights, df_sfdps_flights))

pl.DataFrame(stages)

shape: (4, 5)
┌──────────────────────┬─────────────┬─────────────┬───────────┬─────────────┐
│ stage                ┆ bts_flights ┆ bts_icaos   ┆ sfdps_fli ┆ sfdps_icaos │
│ ---                  ┆ ---         ┆ ---         ┆ ghts      ┆ ---         │
│ str                  ┆ i64         ┆ i64         ┆ ---       ┆ i64         │
│                      ┆             ┆             ┆ i64       ┆             │
╞══════════════════════╪═════════════╪═════════════╪═══════════╪═════════════╡
│ raw                  ┆ 17609       ┆ 4718        ┆ 27486     ┆ 12559       │
│ shared ICAOs         ┆ 17092       ┆ 4441        ┆ 14214     ┆ 4441        │
│ landing date         ┆ 15035       ┆ 4421        ┆ 14214     ┆ 4441        │
│ continental U.S.     ┆ 14702       ┆ 4368        ┆ 13856     ┆ 4380        │
└──────────────────────┴─────────────┴─────────────┴───────────┴─────────────┘

## Require every comparison row for an ICAO to match

The comparison produces one row per paired or unpaired flight. The window expression below retains an ICAO only when every row for it has `match_status == "both"`. Each retained row is therefore one BTS/SFDPS flight pair.

In [3]:
original_columns = df_bts_flights.columns

df_comparison = df_flights_comparision(
    df_bts_flights, df_sfdps_flights, start_dt, end_dt
)
df_both = df_comparison.filter(
    (pl.col("match_status") == "both").all().over("icao")
)

# This is the same BTS-column selection made by bts_and_sfdps_to_gold.py.
df_candidates = df_both.select(original_columns)

pl.DataFrame({
    "result": [
        "comparison rows before filter",
        "retained BTS flights",
        "retained SFDPS flights",
    ],
    "flights": [df_comparison.height, df_both.height, df_both.height],
    "icaos": [df_comparison["icao"].n_unique(), df_both["icao"].n_unique(), df_both["icao"].n_unique()],
})

shape: (3, 3)
┌───────────────────────────────┬─────────┬───────┐
│ result                        ┆ flights ┆ icaos │
│ ---                           ┆ ---     ┆ ---   │
│ str                           ┆ i64     ┆ i64   │
╞═══════════════════════════════╪═════════╪═══════╡
│ comparison rows before filter ┆ 15057   ┆ 4383  │
│ retained BTS flights          ┆ 9852    ┆ 3136  │
│ retained SFDPS flights        ┆ 9852    ┆ 3136  │
└───────────────────────────────┴─────────┴───────┘

## Validate the 9,852 candidate flights against each ADS-B network

`source="adsblol"` is the codebase name for adsb.lol; `source="adsbx"` is ADS-B Exchange.

In [4]:
network_names = {
    "adsblol": "adsb.lol",
    "opensky": "OpenSky",
    "adsbx": "ADS-B Exchange",
}
candidate_icaos = df_candidates["icao"].unique().to_list()
network_results = []
matched_flights_by_network = {}

for source, network in network_names.items():
    df_adsb = read_adsb(
        target_date,
        icaos=candidate_icaos,
        columns=ADSB_MATCHING_COLUMNS,
        source=source,
    )
    df_matched = get_matching_icaos_in_flights(df_candidates, df_adsb, start_dt)
    matched_flights_by_network[source] = df_matched
    network_results.append({
        "network": network,
        "adsb_messages": df_adsb.height,
        "candidate_flights": df_candidates.height,
        "remaining_flights": df_matched.height,
        "remaining_icaos": df_matched["icao"].n_unique(),
        "retained_pct": 100 * df_matched.height / df_candidates.height,
    })

network_summary = pl.DataFrame(network_results)
network_summary

shape: (3, 6)
┌────────────────┬──────────────┬─────────────┬────────────────┬───────────────┬───────────────┐
│ network        ┆ adsb_message ┆ candidate_f ┆ remaining_flig ┆ remaining_ica ┆ retained_pct  │
│ ---            ┆ s            ┆ lights      ┆ hts            ┆ os            ┆ ---           │
│ str            ┆ ---          ┆ ---         ┆ ---            ┆ ---           ┆ f64           │
│                ┆ i64          ┆ i64         ┆ i64            ┆ i64           ┆               │
╞════════════════╪══════════════╪═════════════╪════════════════╪═══════════════╪═══════════════╡
│ adsb.lol       ┆ 11619944     ┆ 9852        ┆ 2313           ┆ 602           ┆ 23.477467     │
│ OpenSky        ┆ 103137616    ┆ 9852        ┆ 458            ┆ 136           ┆ 4.648802      │
│ ADS-B Exchange ┆ 11096574     ┆ 9852        ┆ 2377           ┆ 607           ┆ 24.126066     │
└────────────────┴──────────────┴─────────────┴────────────────┴───────────────┴───────────────┘

## Result

- Raw inputs: **17,609 BTS flights** and **27,486 SFDPS flights**.
- The raw inputs have **4,441 ICAOs in common**. The full preprocessing and all-`both` comparison filter retain **3,136**, so **1,305 of the 4,441 common ICAOs were eliminated (29.39%)**.
- Immediately before comparison: **14,702 BTS flights** and **13,856 SFDPS flights**.
- After requiring all comparison rows for an ICAO to be `both`: **9,852 BTS/SFDPS paired flights** (9,852 from each source) across **3,136 ICAOs**.
- After `get_matching_icaos_in_flights`: **2,313** remain with adsb.lol, **458** with OpenSky, and **2,377** with ADS-B Exchange.

In [5]:
common_icaos_total = len(shared_icaos)
retained_icaos = df_candidates["icao"].n_unique()
eliminated_icaos = common_icaos_total - retained_icaos

pl.DataFrame({
    "common_icaos_total": [common_icaos_total],
    "retained_icaos": [retained_icaos],
    "eliminated_icaos": [eliminated_icaos],
    "eliminated_pct": [100 * eliminated_icaos / common_icaos_total],
})

shape: (1, 4)
┌────────────────────┬────────────────┬──────────────────┬────────────────┐
│ common_icaos_total ┆ retained_icaos ┆ eliminated_icaos ┆ eliminated_pct │
│ ---                ┆ ---            ┆ ---              ┆ ---            │
│ i64                ┆ i64            ┆ i64              ┆ f64            │
╞════════════════════╪════════════════╪══════════════════╪════════════════╡
│ 4441               ┆ 3136           ┆ 1305             ┆ 29.385274      │
└────────────────────┴────────────────┴──────────────────┴────────────────┘